# Siteora Image Model — Dataset Preparation

Build, validate, and inspect the image/caption dataset used for LoRA training.

In [ ]:
import sys
sys.path.insert(0, 'src')


## 1. Load images + captions

Expected layout: a directory of images plus a same-named `.txt` caption per image, OR an already-written `metadata.jsonl`. See `dataset/README.md`.

In [ ]:
from dataset.captions import generate_metadata_from_captions_dir, write_metadata

IMAGES_DIR = 'dataset/raw_images'  # your images + .txt captions
records = generate_metadata_from_captions_dir(IMAGES_DIR)
write_metadata(records, dataset_dir='dataset', metadata_file='metadata.jsonl')
print(f'wrote {len(records)} records')


## 2. Validate images and captions

In [ ]:
from dataset.validation import validate_dataset
report = validate_dataset('dataset')
print(report.summary())


## 3. Create metadata (done above) — inspect a few random pairs

In [ ]:
from dataset.captions import review_captions
for rec in review_captions('dataset', n=8):
    print(rec['file_name'], '->', rec['text'])


## 4. Display random image/caption pairs

In [ ]:
import json, random
from PIL import Image
import matplotlib.pyplot as plt

with open('dataset/metadata.jsonl') as f:
    records = [json.loads(l) for l in f if l.strip()]
sample = random.sample(records, min(6, len(records)))
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, rec in zip(axes.flat, sample):
    img = Image.open(f"dataset/images/{rec['file_name']}")
    ax.imshow(img); ax.set_title(rec['text'][:40] + '...', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()


## 5. Resize/crop to training resolution (writes to a SEPARATE directory)

In [ ]:
from dataset.preprocessing import preprocess_dataset
n = preprocess_dataset('dataset', 'dataset_processed_1024', target_size=1024)
print(f'processed {n} images into dataset_processed_1024/')


## 6. Create train/validation split

In [ ]:
from dataset.preprocessing import train_val_split
train_recs, val_recs = train_val_split('dataset_processed_1024', val_split=0.05)
print(len(train_recs), 'train /', len(val_recs), 'val')


## 7. Dataset statistics

In [ ]:
from dataset.preprocessing import dataset_statistics
print(dataset_statistics('dataset_processed_1024'))
